# Control Adaptativo: Ejemplo

Se realizará el control articular de la trayectoria de un robot de 2 grados de libertad (RR) utilizando control adaptativo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

#%matplotlib notebook

## Simulación del Robot

La siguiente clase simula la dinámica de un robot RR.

In [ ]:
class RobotRR(object):
    """
    Robot RR
    """
    def __init__(self, dt=0.005, X0=None):
        """
        X0 - Estado inicial: [q1, q2, dq1, dq2]
        dt - diferencial de tiempo usado para la integración de Euler
        """
        # Parámetros del robot
        self.m1=1; self.m2=2         # kg
        self.Lc1=0.5; self.Lc2=0.6   # m
        self.L1=1; self.L2=1.2       # m
        self.I1=0.12; self.I2=0.25   # kg.m^2
        self.dt = dt
        self.X = X0

    def set_state(self, X):
        """
        Establece el estado del sistema (q1, q2, dq1, dq2)
        """
        self.X = X

    def get_state(self):
        """
        Recupera el estado actual (q1, q2, dq1, dq2)
        """
        return self.X

    def M_(self, q):
        """
        Retorna la matriz de inercia M(q)
        """
        q1 = q[0]; q2 = q[1]
        m11 = self.m1*self.Lc1**2 + \
              self.m2*(self.L1**2+self.Lc2**2+2*self.L1*self.Lc2*np.cos(q2)) + \
              self.I1 + self.I2
        m12 = self.m2*(self.L1*self.Lc2*np.cos(q2)+self.Lc2**2) + self.I2
        m22 = self.m2*self.Lc2**2 + self.I2
        M = np.array([[m11, m12],[m12, m22]])
        return M

    def C_(self, q, dq):
        """
        Retorna la matriz de Coriolis y fuerza centrífuga C(q,dq)
        """
        q1 = q[0]; q2 = q[1]
        dq1 = dq[0]; dq2 = dq[1]
        h = -self.m2*self.L1*self.Lc2*np.sin(q2)
        C = np.array([[h*dq2, h*(dq1+dq2)],[-h*dq1, 0.0]])
        return C

    def g_(self, q):
        """
        Retorna el vector de gravedad g(q)
        """
        q1 = q[0]; q2 = q[1]
        g = 9.81*np.array([(self.m1*self.Lc1+self.m2*self.L1)*np.cos(q1)+self.m2*self.Lc2*np.cos(q1+q2),
                           self.m2*self.Lc2*np.cos(q1+q2)])
        return g

    def simular(self, u):
        """
        Simulación: retorna los estados X=(q1,q2,dq1,dq2) dado un control u
        """
        # Valores articulares
        q1 = self.X[0]; q2 = self.X[1]
        # Velocidades articulares
        dq1 = self.X[2]; dq2 = self.X[3]

        # Matrices del modelo dinámico
        M = self.M_([q1, q2])
        C = self.C_([q1, q2], [dq1, dq2])
        g = self.g_([q1, q2])

        # Aplicación de torque al robot (actualización del estado)
        ddq = -np.linalg.inv(M).dot( C.dot(np.array([dq1,dq2])) + g - u)
        dq  = np.array([dq1,dq2])
        dX  = np.hstack((dq,ddq))
        # Integración de Euler
        self.X = self.X + self.dt*dX
        return self.X

    def get_params(self):
        """
        Retorna los parámetros dinámicos reales
        """
        pi1 = self.m1*self.Lc1**2 + self.m2*(self.L1**2+self.Lc2**2) + self.I1 + self.I2
        pi2 = self.m2*self.L1*self.Lc2
        pi3 = self.I2 + self.m2*self.Lc2**2
        pi4 = self.m1*self.Lc1 + self.m2*self.L1
        pi5 = self.m2*self.Lc2
        return np.array([pi1, pi2, pi3, pi4, pi5])

## Ley de control

El objetivo de control consiste en el seguimiento de la trayectoria deseada en posición, velocidad y aceleración: $q_d(t) \in \mathbf{R}^2$, $\dot q_d(t) \in \mathbf{R}^2$, $\ddot q_d(t) \in \mathbf{R}^2$. Cada uno es una función del tiempo, pero por simplicidad se omitirá dicha dependencia en la notación.

El control adaptativo consiste en aplicar la siguiente ley de control $u$:
$$
u = Y(q,\dot q, \dot q_r, \ddot q_r) \hat{\pi} - K_d s,
$$

donde $Y \in \mathbb{R}^{2 \times 5}$ es la matriz regresora, $\hat{\pi} \in \mathbb{R}^5$ es el vector de parámetros estimados, $K_d \in \mathbb{R}^{2 \times 2}$ es una matriz de ganancias, y $s = (\dot q - \dot q_d) + \Lambda (q - q_d)$, con $\Lambda  \in \mathbb{R}^{2 \times 2}$ siendo una matriz definida positiva. Además, $\dot q_r$ y $\ddot q_r$ utilizados en $Y$ están dados por $\dot q_r = \dot q_d - \Lambda(q-q_d)$ y $\ddot q_r = \ddot q_d - \Lambda (\dot q-\dot q_d)$.

Los parámetros estimados $\hat{\pi}$ se obtienen a partir de la ley de adaptación dada por:

$$
\dot{\hat{\pi}} = -P~ Y(q,\dot q, \dot q_r, \ddot q_r)^T s
$$

donde $P \in \mathbb{R}^{5 \times 5}$ es una matriz de ponderación (ganancia) para los parámetros.

## Matriz Regresora

Se calculará la matriz $Y(q, \dot q, \dot q_r, \ddot q_r)$, de dimensión $2 \times 5$ para el robot RR. Esta matriz se obtiene de la linealización de los parámetros dinámicos, de tal modo que se cumple:
$$
Y(q, \dot q, \dot q_r, \ddot q_r) = M(q) \ddot q_r + C(q, \dot q) \dot q_r + g(q)
$$

donde $M(q)$ es la matriz de inercia del robot, $C(q,\dot q)$ es la matriz de Coriolis y fuerzas centrífugas, y $g(q)$ es el vector que incluye el efecto de la aceleración de la gravedad. Notar la presencia de los elementos $\ddot q_r$ y $\dot q_r$, y notar que $\ddot q_r \neq \ddot q$ y $\dot q_r \neq \dot q$, por lo que se tiene que adecuar la expresión de $Y$.

In [ ]:
class ControladorAdaptativo:
    def __init__(self, dt, Kd, lambda_, P):
        self.Kd = Kd
        self.lambda_ = lambda_
        self.P = P
        self.dt = dt
        # Estimación inicial de los parámetros dinámicos
        self.pi = np.zeros(5)

    def compute_Y(self, q, dq, dq_r, ddq_r):
        """
        Retorna la matriz Y
        """
        # Extracción de valores de entrada
        q1=q[0]; q2=q[1]; dq1=dq[0]; dq2=dq[1]
        dqr1=dq_r[0]; dqr2=dq_r[1]; ddqr1=ddq_r[0]; ddqr2=ddq_r[1]
        grav = 9.81

        # Elementos de la matriz Y
        y1 = 2*np.cos(q2)*ddqr1 + np.cos(q2)*ddqr2 - np.sin(q2)*dqr1*dq2 - \
             np.sin(q2)*dqr2*(dq1+dq2)
        y2 = np.cos(q2)*ddqr1 + np.sin(q2)*dq1*dqr1

        # Matriz Y
        Y = np.array([[ddqr1, y1, ddqr2, grav*np.cos(q1), grav*np.cos(q1+q2)],
                      [0, y2, ddqr1+ddqr2, 0, grav*np.cos(q1+q2)]])
        return Y

    def get_control(self, q, dq, q_des, dq_des, ddq_des):
        """
        Retorna la señal de control u
        """
        # Errores
        e  = q - q_des
        de = dq - dq_des
        # Términos con r
        dq_r  = dq_des - self.lambda_ @ e
        ddq_r = ddq_des - self.lambda_ @ de
        # Valor de s
        s = de + self.lambda_ @ e

        # Cálculo de Y
        Y = self.compute_Y(q, dq, dq_r, ddq_r)

        # Ley de adaptación
        dpi = -self.P @ Y.T @ s
        # Integración de los parámetros estimados
        self.pi = self.pi + self.dt*dpi
        # Ley de control adaptativo
        u = Y @ self.pi - self.Kd @ s

        return u, self.pi

### Trayectoria de referencia

Se utilizará dos trayectorias de referencia. La primera trayectoria es:
$$
q_d(t) = \begin{bmatrix} 1-\cos(2 \pi t) \\ 2(1-\cos(2 \pi t)) \end{bmatrix}
\qquad \text{y} \qquad
\dot q_d(t) = \begin{bmatrix} 2\pi \sin(2 \pi t) \\ 4 \pi \sin(2 \pi t) \end{bmatrix}
$$

La segunda trayectoria es:
$$
q_d(t) = \begin{bmatrix} 1 - e^{-t} \\ 2(1-e^{-t}) \end{bmatrix}
\qquad \text{y} \qquad
\dot q_d(t) = \begin{bmatrix} e^{-t} \\ 2e^{-t} \end{bmatrix}
$$

In [ ]:
def generar_trayectoria(tfinal, dt, tipo=1):
    """
    Retorna una trayectoria dado el intervalo dt y el tiempo final
    """
    t = np.linspace(0, tfinal, int(tfinal/dt))

    if (tipo == 1):
        q_des   = np.array([1-np.cos(2*np.pi*t), 2-2*np.cos(2*np.pi*t)])
        dq_des  = np.array([2*np.pi*np.sin(2*np.pi*t), 4*np.pi*np.sin(2*np.pi*t)])
        ddq_des = np.array([4*np.pi**2*np.cos(2*np.pi*t), 8*np.pi**2*np.cos(2*np.pi*t)])
    elif (tipo == 2):
        q_des   = np.array([1-np.exp(-t), 2-2*np.exp(-t)])
        dq_des  = np.array([np.exp(-t), 2.*np.exp(-t)])
        ddq_des = np.array([-np.exp(-t), -2.*np.exp(-t)])
        #ddq_des = -0.1*dq_des

    return q_des, dq_des, ddq_des, t

In [ ]:
# Trayectoria generada
tfinal = 5
dt = 0.001
q_des, dq_des, ddq_des, t = generar_trayectoria(tfinal, dt, tipo=1)

# Graficar la trayectoria de referencia
plt.figure(figsize=(10,3))
plt.plot(t, q_des[0,:]); plt.plot(t, q_des[1,:])
plt.title("Trayectorias de referencia")
plt.legend(['$q_1$','$q_2$'], loc='upper right')
plt.xlabel("tiempo [s]"); plt.ylabel("Posición angular [rad]")
plt.grid()

## Simulación

In [ ]:
# Número de iteraciones
N = int(tfinal/dt)
# Variables para almacenar estados y torques
Xt = np.zeros((4,N))   # Almacena estados
Ut = np.zeros((2,N))   # Almacena las señales de control
Pi = np.zeros((5,N))   # Almacena los parámetros estimados

# Robot simulado
robot = RobotRR(dt)

# Ganancias
Kd      = 50*np.eye(2)
lambda_ = 10*np.eye(2)
P       = 1.0*np.diag([0.3, 0.3, 0.3, 0.3, 0.3])
# Controlador
controlador = ControladorAdaptativo(dt, Kd, lambda_, P)

**Bucle de simulación principal**

In [ ]:
# Condiciones iniciales: [q1, q2, dq1, dq2]
X0 = np.array([0.0, 0.0, 0.0, 0.0])
X = np.copy(X0)
robot.set_state(X0)

# Bucle de simulación
for i in range(N):

    # Tiempo actual
    t = dt*i
    # Valores articulares
    q  = X[0:2]; dq = X[2:]

    # Controlador
    u, pi = controlador.get_control(q, dq, q_des[:,i], dq_des[:,i], ddq_des[:,i])
    # Simulación de movimiento del robot
    X = robot.simular(u)

    # Almacenamiento de estados y control
    Xt[:,i] = X
    Ut[:,i] = u
    Pi[:,i] = pi

## Resultados

Las siguientes figuras muestran los valores de torque ($u_1, u_2$) aplicados a cada articulación usando la ley de control.

In [ ]:
# Vector de tiempo
t = dt*np.arange(N)
# Vectores de torque articular
u1 = Ut[0,:]; u2 = Ut[1,:]

# Gráfico de torques
plt.figure(figsize=(12,3))
# Torque u1 en la articulación q1
plt.subplot(1,2,1)
plt.plot(t,u1); plt.grid(); plt.title("Torque $u_1$"); plt.xlabel("tiempo [s]")
# Torque u2 en la articulación q2
plt.subplot(1,2,2)
plt.plot(t,u2); plt.grid(); plt.title("Torque $u_2$"); plt.xlabel("tiempo [s]")
plt.show()

Trayectoria articular deseada $q_{d}(t)$ y trayectoria articular obtenida $q(t)$ para cada una de las articulaciones. Igualmente, se muestra la velocidad articular deseada $\dot q_d(t)$ y la velocidad articular obtenida $\dot q(t)$ en cada articulación.

In [ ]:
# Trayectoria articular y trayectoria deseada
plt.figure(figsize=(12,6))

# Trayectoria para q1
plt.subplot(2,2,1)
plt.plot(t,Xt[0,:]); plt.plot(t, q_des[0,:]); plt.grid()
plt.title("Articulación 1"); plt.xlabel("tiempo [s]"); plt.legend(['$q_1$','$q_{1des}$'])
# Trayectoria para q2
plt.subplot(2,2,2)
plt.plot(t,Xt[1,:]); plt.plot(t, q_des[1,:]); plt.grid()
plt.title("Articulación 2"); plt.xlabel("tiempo [s]"); plt.legend(['$q_2$','$q_{2des}$'])
plt.show()

# Velocidad articular y velocidad deseada
plt.figure(figsize=(12,6))

# Velocidad articular para q1
plt.subplot(2,2,3)
plt.plot(t,Xt[2,:]); plt.plot(t,dq_des[0,:]); plt.grid()
plt.title("Velocidad en articulación 1"); plt.xlabel("tiempo [s]"); plt.legend(['$dq_1$','$dq_{1des}$'])
# Velocidad articular para q2
plt.subplot(2,2,4)
plt.plot(t,Xt[3,:]); plt.plot(t,dq_des[1,:]); plt.grid()
plt.title("Velocidad en articulación 2"); plt.xlabel("tiempo [s]"); plt.legend(['$dq_2$','$dq_{2des}$'])

plt.show()

Errores en la posición articular $e(t)=q_d(t) - q(t)$, y errores en las velocidades articulares $\dot e = \dot q_d(t) - \dot q(t)$.

In [ ]:
# Gráfico de errores
plt.figure(figsize=(12,3))

# Error en las posiciones articulares (deseado - obtenido)
plt.subplot(1,2,1)
plt.plot(t, q_des[0,:]-Xt[0,:]); plt.plot(t, q_des[1,:]-Xt[1,:]); plt.grid()
plt.title("Errores articulares"); plt.legend(['$e_{1}$', '$e_{2}$'])

# Error en las velocidades articulares (deseado - obtenido)
plt.subplot(1,2,2)
plt.plot(t, dq_des[0,:]-Xt[2,:]); plt.plot(t, dq_des[1,:]-Xt[3,:]); plt.grid()
plt.title("Errores en velocidades articulares"); plt.legend(['$de_{1}$', '$de_{2}$'])

plt.show()

Las siguientes figuras muestran en líneas sólidas los parámetros dinámicos estimados $\hat{\pi}$ y en líneas punteadas los parámetros dinámicos reales $\pi$. Cada parámetro se muestra con un color diferente.

In [ ]:
pi_real = robot.get_params()

plt.plot(t,Pi[0,:],'r'); plt.plot(t,pi_real[0]*np.ones(t.size),'r--')
plt.plot(t,Pi[1,:],'g'); plt.plot(t,pi_real[1]*np.ones(t.size),'g--')
plt.plot(t,Pi[2,:],'b'); plt.plot(t,pi_real[2]*np.ones(t.size),'b--')
plt.plot(t,Pi[3,:],'c'); plt.plot(t,pi_real[3]*np.ones(t.size),'c--')
plt.plot(t,Pi[4,:],'y'); plt.plot(t,pi_real[4]*np.ones(t.size),'y--')
plt.show()